# 06 · Metadata oficial y ground truth proxy de roles

Cruza la transcripción de Whisper (Notebook 05) con la transcripción oficial de BigQuery para inferir, sin etiquetas manuales, qué speaker es AGENT y cuál CLIENT en cada llamada.

Pasos visibles (estilo Notebook 01): cargar datos, cruzar con metadata, extraer turnos oficiales, **alinear texto (paso pesado con checkpoint)**, mapear speaker→rol por evidencia agregada, propagar el rol a los segmentos, evaluar en holdout y publicar.

El único paso pesado (recorre audios) es la alineación textual; solo ese guarda checkpoints. Todo lo demás son transformaciones sobre tablas y se sube a GCS al final.

In [1]:
# INSTALACIÓN DE REQUISITOS PREVIOS
from pathlib import Path

CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
requirements_path = REPO_ROOT / "requirements.txt"

%pip install -q -r {requirements_path}

print("Requisitos instalados correctamente.")

Note: you may need to restart the kernel to use updated packages.
Requisitos instalados correctamente.


In [2]:
# IMPORTS Y ACCESO AL REPOSITORIO
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.cloud import storage, bigquery
from IPython.display import display, clear_output

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import (
    DATA_DIR,
    OUTPUT_DIR,
    CONSOLIDATED_ALL_FINAL_MERGED_SEGMENTS_CSV,
    TRANSCRIPTION_FINAL_SEGMENTS_CSV,
    BQ_PROJECT_ID, BQ_DATASET, BQ_METADATA_SOURCES,
    PROXY_GROUNDTRUTH_DIR, PROXY_CHECKPOINT_DIR,
    PROXY_SEGMENT_LEVEL_CSV, GCS_PROXY_GROUNDTRUTH_PREFIX,
    ensure_phase06_directories,
)
from src import metadata_groundtruth_proxy as mgp

gcs_client = storage.Client()
ensure_phase06_directories()
pd.set_option("display.max_columns", 120)
print("Repositorio y cliente GCS configurados.")

Repositorio y cliente GCS configurados.


In [3]:
# CONFIGURACIÓN DE LA FASE 06
# Todos los parámetros ajustables viven aquí. Cambia cualquier número sin
# tocar los archivos .py.

# --- Control de ejecución ---
FORCE_PROXY = False               # True = recalcula aunque exista el output final
MAX_AUDIOS_TO_PROCESS = None      # None = todos; un número limita la alineación
SAVE_CHECKPOINT_EVERY_N_AUDIOS = 50
UPLOAD_TO_GCS = True

# --- Ventanas y turnos de alineación textual ---
MAX_WINDOW_SIZE = 4
MAX_GAP_BETWEEN_SEGMENTS_SEC = 1.25
MIN_WORDS_OFFICIAL = 4
MIN_WORDS_WHISPER_WINDOW = 4
MIN_TEXT_CHARS = 18

# --- Aceptación de un match textual ---
ACCEPT_COMBINED_SCORE = 0.70
ACCEPT_CHAR_COSINE = 0.50
ACCEPT_TOKEN_CONTAINMENT = 0.35
ACCEPT_MARGIN = 0.03

# --- Reglas de mapeo speaker -> rol ---
MIN_MATCHES_FOR_SPEAKER_ROLE = 2
MIN_ROLE_PURITY = 0.70
REQUIRE_BOTH_ROLES_PER_AUDIO = True
REQUIRE_ONE_TO_ONE_AGENT_CLIENT_MAPPING = True
MIN_MATCHES_PER_ROLE_IN_AUDIO = 1

# --- Evidencia textual agregada por speaker ---
ROLE_EVIDENCE_MIN_WORDS_PER_SEGMENT = 3
ROLE_EVIDENCE_MAX_OVERLAP_RATIO_STRICT = 0.05
ROLE_EVIDENCE_MIN_SEGMENTS_PER_SPEAKER = 2
ROLE_EVIDENCE_MIN_SECONDS_PER_SPEAKER = 6.0
ROLE_EVIDENCE_MIN_WORDS_PER_SPEAKER = 15
ROLE_EVIDENCE_MAX_SEGMENTS_PER_SPEAKER = 10
ROLE_EVIDENCE_TARGET_SECONDS_PER_SPEAKER = 25.0

# --- Aceptación del par agregado ---
MIN_AGGREGATED_PAIR_MARGIN = 0.01
MIN_AGGREGATED_PAIR_SCORE = 0.10

# Diccionario de parámetros que consumen las funciones del módulo.
PARAMS = {
    "MAX_WINDOW_SIZE": MAX_WINDOW_SIZE,
    "MAX_GAP_BETWEEN_SEGMENTS_SEC": MAX_GAP_BETWEEN_SEGMENTS_SEC,
    "MIN_WORDS_WHISPER_WINDOW": MIN_WORDS_WHISPER_WINDOW,
    "ACCEPT_COMBINED_SCORE": ACCEPT_COMBINED_SCORE,
    "ACCEPT_CHAR_COSINE": ACCEPT_CHAR_COSINE,
    "ACCEPT_TOKEN_CONTAINMENT": ACCEPT_TOKEN_CONTAINMENT,
    "ACCEPT_MARGIN": ACCEPT_MARGIN,
    "MIN_MATCHES_FOR_SPEAKER_ROLE": MIN_MATCHES_FOR_SPEAKER_ROLE,
    "MIN_ROLE_PURITY": MIN_ROLE_PURITY,
    "REQUIRE_BOTH_ROLES_PER_AUDIO": REQUIRE_BOTH_ROLES_PER_AUDIO,
    "REQUIRE_ONE_TO_ONE_AGENT_CLIENT_MAPPING": REQUIRE_ONE_TO_ONE_AGENT_CLIENT_MAPPING,
    "MIN_MATCHES_PER_ROLE_IN_AUDIO": MIN_MATCHES_PER_ROLE_IN_AUDIO,
    "ROLE_EVIDENCE_MIN_WORDS_PER_SEGMENT": ROLE_EVIDENCE_MIN_WORDS_PER_SEGMENT,
    "ROLE_EVIDENCE_MAX_OVERLAP_RATIO_STRICT": ROLE_EVIDENCE_MAX_OVERLAP_RATIO_STRICT,
    "ROLE_EVIDENCE_MIN_SEGMENTS_PER_SPEAKER": ROLE_EVIDENCE_MIN_SEGMENTS_PER_SPEAKER,
    "ROLE_EVIDENCE_MIN_SECONDS_PER_SPEAKER": ROLE_EVIDENCE_MIN_SECONDS_PER_SPEAKER,
    "ROLE_EVIDENCE_MIN_WORDS_PER_SPEAKER": ROLE_EVIDENCE_MIN_WORDS_PER_SPEAKER,
    "ROLE_EVIDENCE_MAX_SEGMENTS_PER_SPEAKER": ROLE_EVIDENCE_MAX_SEGMENTS_PER_SPEAKER,
    "ROLE_EVIDENCE_TARGET_SECONDS_PER_SPEAKER": ROLE_EVIDENCE_TARGET_SECONDS_PER_SPEAKER,
    "MIN_AGGREGATED_PAIR_MARGIN": MIN_AGGREGATED_PAIR_MARGIN,
    "MIN_AGGREGATED_PAIR_SCORE": MIN_AGGREGATED_PAIR_SCORE,
}

# Candidatos de ruta para la transcripción de Whisper (Notebook 05).
WHISPER_CANDIDATES = [
    TRANSCRIPTION_FINAL_SEGMENTS_CSV,
    DATA_DIR / "transcription_outputs" / "all_segments_transcribed.csv",
]

print("Configuración de la fase 06 cargada.")
print("Umbral de aceptación combinado:", ACCEPT_COMBINED_SCORE)
print("Checkpoint cada N audios:", SAVE_CHECKPOINT_EVERY_N_AUDIOS)

Configuración de la fase 06 cargada.
Umbral de aceptación combinado: 0.7
Checkpoint cada N audios: 50


In [4]:
# RESTAURACIÓN DESDE GCS Y SALTO DE FASE COMPLETA

mgp.restore_phase_outputs_from_gcs(
    gcs_client, PROXY_GROUNDTRUTH_DIR, GCS_PROXY_GROUNDTRUTH_PREFIX, DATA_DIR
)

SKIP_PHASE = mgp.phase_outputs_complete(PROXY_SEGMENT_LEVEL_CSV) and not FORCE_PROXY

if SKIP_PHASE:
    segment_level_proxy_textual = pd.read_csv(PROXY_SEGMENT_LEVEL_CSV)
    print("Output final ya existe. No se recalcula la fase.")
    print("Segmentos con proxy:", int(segment_level_proxy_textual['official_role_proxy'].notna().sum()))
else:
    print("Se ejecutará la fase 06 completa.")

Restauración desde GCS completada.
Archivos encontrados: 0
Archivos descargados: 0
Archivos locales ya vigentes: 0
Output final ya existe. No se recalcula la fase.
Segmentos con proxy: 39868


In [5]:
# CARGA DE SEGMENTOS, WHISPER Y METADATA OFICIAL (BigQuery)

if not SKIP_PHASE:
    df_segments, df_whisper, whisper_path = mgp.load_segments_and_whisper(
        CONSOLIDATED_ALL_FINAL_MERGED_SEGMENTS_CSV, WHISPER_CANDIDATES
    )
    print("Segmentos:", df_segments.shape, "| Whisper:", df_whisper.shape, f"({whisper_path.name})")

    bq_client = bigquery.Client(project=BQ_PROJECT_ID)
    df_meta = mgp.load_official_metadata_from_bigquery(
        bq_client, BQ_PROJECT_ID, BQ_DATASET, BQ_METADATA_SOURCES, OUTPUT_DIR
    )
    print("Metadata oficial:", df_meta.shape)
    print("Filas con transcripción oficial:", int(df_meta['has_official_transcription'].sum()))

In [6]:
# MATCH AUDIO DIARIZADO <-> METADATA + EXTRACCIÓN DE TURNOS OFICIALES

if not SKIP_PHASE:
    seg_match_audit, coverage = mgp.match_audio_to_metadata(df_segments, df_meta)
    print(f"Cobertura de metadata: {coverage:.1%}")
    print("Audios con metadata:", int(seg_match_audit['metadata_matched'].sum()))
    if coverage < 0.50:
        print("ADVERTENCIA: cobertura < 50%. Los resultados solo cubrirán los audios cruzados.")

    (df_official_turns, df_official_turns_all,
     role_counts_all, eligible_audios) = mgp.extract_official_turns(
        seg_match_audit, MIN_WORDS_OFFICIAL, MIN_TEXT_CHARS, REQUIRE_BOTH_ROLES_PER_AUDIO
    )
    print("\nTurnos oficiales usables:", df_official_turns.shape)
    print("Audios con AGENT y CLIENT:", len(eligible_audios))
    display(df_official_turns_all['official_role'].value_counts())

In [7]:
# ALINEACIÓN TEXTUAL WHISPER <-> OFICIAL (paso pesado con checkpoints)

if not SKIP_PHASE:
    def _progress(i, total, audio_file):
        if i % 10 == 0 or i == 1 or i == total:
            print(f"Alineando {i}/{total}: {audio_file}", flush=True)

    text_alignment_candidates, alignment_processing_summary = mgp.run_text_alignment(
        df_official_turns, df_whisper, PARAMS, PROXY_CHECKPOINT_DIR,
        save_checkpoint_every_n=SAVE_CHECKPOINT_EVERY_N_AUDIOS,
        max_audios_to_process=MAX_AUDIOS_TO_PROCESS,
        progress_callback=_progress,
    )
    print("\nCandidatos:", text_alignment_candidates.shape)

    accepted_matches = mgp.select_accepted_matches(text_alignment_candidates)
    threshold_sensitivity = mgp.compute_threshold_sensitivity(text_alignment_candidates)
    print("Matches aceptados:", accepted_matches.shape)
    if len(accepted_matches):
        display(accepted_matches['official_role'].value_counts())

In [8]:
# MAPEO SPEAKER -> ROL POR EVIDENCIA TEXTUAL AGREGADA (principal)

if not SKIP_PHASE:
    speaker_role_mapping_strict_matches = mgp.infer_one_to_one_speaker_role_mapping_strict(
        accepted_matches, PARAMS
    ) if len(accepted_matches) else pd.DataFrame()

    (speaker_role_mapping, speaker_role_evidence_aggregated,
     speaker_role_score_matrix, speaker_role_mapping_audit) = mgp.infer_aggregated_speaker_role_mapping(
        df_whisper, df_official_turns, PARAMS
    )
    speaker_role_mapping = mgp.enrich_mapping_with_metadata(speaker_role_mapping, seg_match_audit)

    print("Mapping agregado:", speaker_role_mapping.shape)
    if len(speaker_role_mapping_audit):
        display(speaker_role_mapping_audit['aggregate_mapping_status'].value_counts())
    if len(speaker_role_mapping):
        display(speaker_role_mapping['probable_role'].value_counts())

In [9]:
# PROPAGACIÓN DEL ROL A LOS SEGMENTOS + EVALUACIÓN HOLDOUT

if not SKIP_PHASE:
    segment_level_proxy_textual = mgp.propagate_role_to_segments(df_segments, speaker_role_mapping)
    print("Segmentos con proxy:", int(segment_level_proxy_textual['official_role_proxy'].notna().sum()))
    print("Audios con proxy:", segment_level_proxy_textual.loc[segment_level_proxy_textual['official_role_proxy'].notna(), 'audio_file'].nunique())

    holdout_metrics, holdout_predictions, cm_df = mgp.evaluate_mapping_holdout(accepted_matches, PARAMS)
    if len(cm_df):
        print("\nMatriz de confusión holdout:")
        display(cm_df)
    if len(holdout_metrics):
        display(holdout_metrics)

In [10]:
# RESUMEN, GUARDADO LOCAL Y SUBIDA FINAL A GCS

if not SKIP_PHASE:
    textual_proxy_metrics_summary = mgp.build_textual_proxy_summary(
        df_segments, df_whisper, df_official_turns, text_alignment_candidates,
        accepted_matches, speaker_role_mapping_strict_matches, speaker_role_mapping,
        segment_level_proxy_textual,
    )
    display(textual_proxy_metrics_summary)

    result_frames = {
        "segment_level_proxy_groundtruth": segment_level_proxy_textual,
        "speaker_role_mapping_textual": speaker_role_mapping,
        "speaker_role_mapping_strict_matches": speaker_role_mapping_strict_matches,
        "speaker_role_evidence_aggregated": speaker_role_evidence_aggregated,
        "speaker_role_score_matrix_aggregated": speaker_role_score_matrix,
        "speaker_role_mapping_audit_aggregated": speaker_role_mapping_audit,
        "text_alignment_candidates": text_alignment_candidates,
        "text_alignment_matches": accepted_matches,
        "text_alignment_threshold_sensitivity": threshold_sensitivity,
        "textual_proxy_metrics_summary": textual_proxy_metrics_summary,
        "alignment_processing_summary": alignment_processing_summary,
        "holdout_role_mapping_metrics": holdout_metrics,
        "holdout_role_mapping_predictions": holdout_predictions,
        "metadata_join_audit_by_audio": seg_match_audit,
    }
    saved = mgp.save_and_upload_outputs(
        result_frames, PROXY_GROUNDTRUTH_DIR, gcs_client,
        GCS_PROXY_GROUNDTRUTH_PREFIX, upload_to_gcs=UPLOAD_TO_GCS,
    )
    print("\nArchivos guardados:", len(saved))
    print("Fase 06 completada.")
else:
    print("Fase 06 reutilizada desde outputs previos.")

# Incondicional: asegura que lo que hay en local también esté en GCS, tanto
# si se ejecutó la fase como si se saltó por tener outputs locales completos
# (caso típico: un compañero corrió esto en local y nunca lo subió).
mgp.ensure_phase_outputs_in_gcs(gcs_client, PROXY_GROUNDTRUTH_DIR, GCS_PROXY_GROUNDTRUTH_PREFIX)

Fase 06 reutilizada desde outputs previos.
Subiendo outputs 3/49: bq_metadata_notebook00_exact_with_transcripcion.csv
Subiendo outputs 4/49: checkpoints/text_alignment_candidates_checkpoint_0050.csv
Subiendo outputs 5/49: checkpoints/text_alignment_candidates_checkpoint_0100.csv
Subiendo outputs 6/49: checkpoints/text_alignment_candidates_checkpoint_0150.csv
Subiendo outputs 7/49: checkpoints/text_alignment_candidates_checkpoint_0200.csv
Subiendo outputs 8/49: checkpoints/text_alignment_candidates_checkpoint_0250.csv
Subiendo outputs 9/49: checkpoints/text_alignment_candidates_checkpoint_0300.csv
Subiendo outputs 10/49: checkpoints/text_alignment_candidates_checkpoint_0350.csv
Subiendo outputs 11/49: checkpoints/text_alignment_candidates_checkpoint_0400.csv
Subiendo outputs 12/49: checkpoints/text_alignment_candidates_checkpoint_0450.csv
Subiendo outputs 13/49: checkpoints/text_alignment_candidates_checkpoint_0500.csv
Subiendo outputs 14/49: checkpoints/text_alignment_candidates_checkp

# 06 · Metadata oficial y ground truth proxy de roles

Cruza la transcripción de Whisper (Notebook 05) con la transcripción oficial de BigQuery para inferir, sin etiquetas manuales, qué speaker es AGENT y cuál CLIENT en cada llamada.

Pasos visibles (estilo Notebook 01): cargar datos, cruzar con metadata, extraer turnos oficiales, **alinear texto (paso pesado con checkpoint)**, mapear speaker→rol por evidencia agregada, propagar el rol a los segmentos, evaluar en holdout y publicar.

El único paso pesado (recorre audios) es la alineación textual; solo ese guarda checkpoints. Todo lo demás son transformaciones sobre tablas y se sube a GCS al final.

In [1]:
# INSTALACIÓN DE REQUISITOS PREVIOS
from pathlib import Path

CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
requirements_path = REPO_ROOT / "requirements.txt"

%pip install -q -r {requirements_path}

print("Requisitos instalados correctamente.")

Note: you may need to restart the kernel to use updated packages.
Requisitos instalados correctamente.


In [2]:
# IMPORTS Y ACCESO AL REPOSITORIO
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.cloud import storage, bigquery
from IPython.display import display, clear_output

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import (
    DATA_DIR,
    OUTPUT_DIR,
    CONSOLIDATED_ALL_FINAL_MERGED_SEGMENTS_CSV,
    TRANSCRIPTION_FINAL_SEGMENTS_CSV,
    BQ_PROJECT_ID, BQ_DATASET, BQ_METADATA_SOURCES,
    PROXY_GROUNDTRUTH_DIR, PROXY_CHECKPOINT_DIR,
    PROXY_SEGMENT_LEVEL_CSV, GCS_PROXY_GROUNDTRUTH_PREFIX,
    ensure_phase06_directories,
)
from src import metadata_groundtruth_proxy as mgp

gcs_client = storage.Client()
ensure_phase06_directories()
pd.set_option("display.max_columns", 120)
print("Repositorio y cliente GCS configurados.")

Repositorio y cliente GCS configurados.


In [3]:
# CONFIGURACIÓN DE LA FASE 06
# Todos los parámetros ajustables viven aquí. Cambia cualquier número sin
# tocar los archivos .py.

# --- Control de ejecución ---
FORCE_PROXY = False               # True = recalcula aunque exista el output final
MAX_AUDIOS_TO_PROCESS = None      # None = todos; un número limita la alineación
SAVE_CHECKPOINT_EVERY_N_AUDIOS = 50
UPLOAD_TO_GCS = True

# --- Ventanas y turnos de alineación textual ---
MAX_WINDOW_SIZE = 4
MAX_GAP_BETWEEN_SEGMENTS_SEC = 1.25
MIN_WORDS_OFFICIAL = 4
MIN_WORDS_WHISPER_WINDOW = 4
MIN_TEXT_CHARS = 18

# --- Aceptación de un match textual ---
ACCEPT_COMBINED_SCORE = 0.70
ACCEPT_CHAR_COSINE = 0.50
ACCEPT_TOKEN_CONTAINMENT = 0.35
ACCEPT_MARGIN = 0.03

# --- Reglas de mapeo speaker -> rol ---
MIN_MATCHES_FOR_SPEAKER_ROLE = 2
MIN_ROLE_PURITY = 0.70
REQUIRE_BOTH_ROLES_PER_AUDIO = True
REQUIRE_ONE_TO_ONE_AGENT_CLIENT_MAPPING = True
MIN_MATCHES_PER_ROLE_IN_AUDIO = 1

# --- Evidencia textual agregada por speaker ---
ROLE_EVIDENCE_MIN_WORDS_PER_SEGMENT = 3
ROLE_EVIDENCE_MAX_OVERLAP_RATIO_STRICT = 0.05
ROLE_EVIDENCE_MIN_SEGMENTS_PER_SPEAKER = 2
ROLE_EVIDENCE_MIN_SECONDS_PER_SPEAKER = 6.0
ROLE_EVIDENCE_MIN_WORDS_PER_SPEAKER = 15
ROLE_EVIDENCE_MAX_SEGMENTS_PER_SPEAKER = 10
ROLE_EVIDENCE_TARGET_SECONDS_PER_SPEAKER = 25.0

# --- Aceptación del par agregado ---
MIN_AGGREGATED_PAIR_MARGIN = 0.01
MIN_AGGREGATED_PAIR_SCORE = 0.10

# Diccionario de parámetros que consumen las funciones del módulo.
PARAMS = {
    "MAX_WINDOW_SIZE": MAX_WINDOW_SIZE,
    "MAX_GAP_BETWEEN_SEGMENTS_SEC": MAX_GAP_BETWEEN_SEGMENTS_SEC,
    "MIN_WORDS_WHISPER_WINDOW": MIN_WORDS_WHISPER_WINDOW,
    "ACCEPT_COMBINED_SCORE": ACCEPT_COMBINED_SCORE,
    "ACCEPT_CHAR_COSINE": ACCEPT_CHAR_COSINE,
    "ACCEPT_TOKEN_CONTAINMENT": ACCEPT_TOKEN_CONTAINMENT,
    "ACCEPT_MARGIN": ACCEPT_MARGIN,
    "MIN_MATCHES_FOR_SPEAKER_ROLE": MIN_MATCHES_FOR_SPEAKER_ROLE,
    "MIN_ROLE_PURITY": MIN_ROLE_PURITY,
    "REQUIRE_BOTH_ROLES_PER_AUDIO": REQUIRE_BOTH_ROLES_PER_AUDIO,
    "REQUIRE_ONE_TO_ONE_AGENT_CLIENT_MAPPING": REQUIRE_ONE_TO_ONE_AGENT_CLIENT_MAPPING,
    "MIN_MATCHES_PER_ROLE_IN_AUDIO": MIN_MATCHES_PER_ROLE_IN_AUDIO,
    "ROLE_EVIDENCE_MIN_WORDS_PER_SEGMENT": ROLE_EVIDENCE_MIN_WORDS_PER_SEGMENT,
    "ROLE_EVIDENCE_MAX_OVERLAP_RATIO_STRICT": ROLE_EVIDENCE_MAX_OVERLAP_RATIO_STRICT,
    "ROLE_EVIDENCE_MIN_SEGMENTS_PER_SPEAKER": ROLE_EVIDENCE_MIN_SEGMENTS_PER_SPEAKER,
    "ROLE_EVIDENCE_MIN_SECONDS_PER_SPEAKER": ROLE_EVIDENCE_MIN_SECONDS_PER_SPEAKER,
    "ROLE_EVIDENCE_MIN_WORDS_PER_SPEAKER": ROLE_EVIDENCE_MIN_WORDS_PER_SPEAKER,
    "ROLE_EVIDENCE_MAX_SEGMENTS_PER_SPEAKER": ROLE_EVIDENCE_MAX_SEGMENTS_PER_SPEAKER,
    "ROLE_EVIDENCE_TARGET_SECONDS_PER_SPEAKER": ROLE_EVIDENCE_TARGET_SECONDS_PER_SPEAKER,
    "MIN_AGGREGATED_PAIR_MARGIN": MIN_AGGREGATED_PAIR_MARGIN,
    "MIN_AGGREGATED_PAIR_SCORE": MIN_AGGREGATED_PAIR_SCORE,
}

# Candidatos de ruta para la transcripción de Whisper (Notebook 05).
WHISPER_CANDIDATES = [
    TRANSCRIPTION_FINAL_SEGMENTS_CSV,
    DATA_DIR / "transcription_outputs" / "all_segments_transcribed.csv",
]

print("Configuración de la fase 06 cargada.")
print("Umbral de aceptación combinado:", ACCEPT_COMBINED_SCORE)
print("Checkpoint cada N audios:", SAVE_CHECKPOINT_EVERY_N_AUDIOS)

Configuración de la fase 06 cargada.
Umbral de aceptación combinado: 0.7
Checkpoint cada N audios: 50


In [4]:
# RESTAURACIÓN DESDE GCS Y SALTO DE FASE COMPLETA

mgp.restore_phase_outputs_from_gcs(
    gcs_client, PROXY_GROUNDTRUTH_DIR, GCS_PROXY_GROUNDTRUTH_PREFIX, DATA_DIR
)

SKIP_PHASE = mgp.phase_outputs_complete(PROXY_SEGMENT_LEVEL_CSV) and not FORCE_PROXY

if SKIP_PHASE:
    segment_level_proxy_textual = pd.read_csv(PROXY_SEGMENT_LEVEL_CSV)
    print("Output final ya existe. No se recalcula la fase.")
    print("Segmentos con proxy:", int(segment_level_proxy_textual['official_role_proxy'].notna().sum()))
else:
    print("Se ejecutará la fase 06 completa.")

Restauración desde GCS completada.
Archivos encontrados: 0
Archivos descargados: 0
Archivos locales ya vigentes: 0
Output final ya existe. No se recalcula la fase.
Segmentos con proxy: 39868


In [5]:
# CARGA DE SEGMENTOS, WHISPER Y METADATA OFICIAL (BigQuery)

if not SKIP_PHASE:
    df_segments, df_whisper, whisper_path = mgp.load_segments_and_whisper(
        CONSOLIDATED_ALL_FINAL_MERGED_SEGMENTS_CSV, WHISPER_CANDIDATES
    )
    print("Segmentos:", df_segments.shape, "| Whisper:", df_whisper.shape, f"({whisper_path.name})")

    bq_client = bigquery.Client(project=BQ_PROJECT_ID)
    df_meta = mgp.load_official_metadata_from_bigquery(
        bq_client, BQ_PROJECT_ID, BQ_DATASET, BQ_METADATA_SOURCES, OUTPUT_DIR
    )
    print("Metadata oficial:", df_meta.shape)
    print("Filas con transcripción oficial:", int(df_meta['has_official_transcription'].sum()))

In [6]:
# MATCH AUDIO DIARIZADO <-> METADATA + EXTRACCIÓN DE TURNOS OFICIALES

if not SKIP_PHASE:
    seg_match_audit, coverage = mgp.match_audio_to_metadata(df_segments, df_meta)
    print(f"Cobertura de metadata: {coverage:.1%}")
    print("Audios con metadata:", int(seg_match_audit['metadata_matched'].sum()))
    if coverage < 0.50:
        print("ADVERTENCIA: cobertura < 50%. Los resultados solo cubrirán los audios cruzados.")

    (df_official_turns, df_official_turns_all,
     role_counts_all, eligible_audios) = mgp.extract_official_turns(
        seg_match_audit, MIN_WORDS_OFFICIAL, MIN_TEXT_CHARS, REQUIRE_BOTH_ROLES_PER_AUDIO
    )
    print("\nTurnos oficiales usables:", df_official_turns.shape)
    print("Audios con AGENT y CLIENT:", len(eligible_audios))
    display(df_official_turns_all['official_role'].value_counts())

In [7]:
# ALINEACIÓN TEXTUAL WHISPER <-> OFICIAL (paso pesado con checkpoints)

if not SKIP_PHASE:
    def _progress(i, total, audio_file):
        if i % 10 == 0 or i == 1 or i == total:
            print(f"Alineando {i}/{total}: {audio_file}", flush=True)

    text_alignment_candidates, alignment_processing_summary = mgp.run_text_alignment(
        df_official_turns, df_whisper, PARAMS, PROXY_CHECKPOINT_DIR,
        save_checkpoint_every_n=SAVE_CHECKPOINT_EVERY_N_AUDIOS,
        max_audios_to_process=MAX_AUDIOS_TO_PROCESS,
        progress_callback=_progress,
    )
    print("\nCandidatos:", text_alignment_candidates.shape)

    accepted_matches = mgp.select_accepted_matches(text_alignment_candidates)
    threshold_sensitivity = mgp.compute_threshold_sensitivity(text_alignment_candidates)
    print("Matches aceptados:", accepted_matches.shape)
    if len(accepted_matches):
        display(accepted_matches['official_role'].value_counts())

In [8]:
# MAPEO SPEAKER -> ROL POR EVIDENCIA TEXTUAL AGREGADA (principal)

if not SKIP_PHASE:
    speaker_role_mapping_strict_matches = mgp.infer_one_to_one_speaker_role_mapping_strict(
        accepted_matches, PARAMS
    ) if len(accepted_matches) else pd.DataFrame()

    (speaker_role_mapping, speaker_role_evidence_aggregated,
     speaker_role_score_matrix, speaker_role_mapping_audit) = mgp.infer_aggregated_speaker_role_mapping(
        df_whisper, df_official_turns, PARAMS
    )
    speaker_role_mapping = mgp.enrich_mapping_with_metadata(speaker_role_mapping, seg_match_audit)

    print("Mapping agregado:", speaker_role_mapping.shape)
    if len(speaker_role_mapping_audit):
        display(speaker_role_mapping_audit['aggregate_mapping_status'].value_counts())
    if len(speaker_role_mapping):
        display(speaker_role_mapping['probable_role'].value_counts())

In [9]:
# PROPAGACIÓN DEL ROL A LOS SEGMENTOS + EVALUACIÓN HOLDOUT

if not SKIP_PHASE:
    segment_level_proxy_textual = mgp.propagate_role_to_segments(df_segments, speaker_role_mapping)
    print("Segmentos con proxy:", int(segment_level_proxy_textual['official_role_proxy'].notna().sum()))
    print("Audios con proxy:", segment_level_proxy_textual.loc[segment_level_proxy_textual['official_role_proxy'].notna(), 'audio_file'].nunique())

    holdout_metrics, holdout_predictions, cm_df = mgp.evaluate_mapping_holdout(accepted_matches, PARAMS)
    if len(cm_df):
        print("\nMatriz de confusión holdout:")
        display(cm_df)
    if len(holdout_metrics):
        display(holdout_metrics)

In [10]:
# RESUMEN, GUARDADO LOCAL Y SUBIDA FINAL A GCS

if not SKIP_PHASE:
    textual_proxy_metrics_summary = mgp.build_textual_proxy_summary(
        df_segments, df_whisper, df_official_turns, text_alignment_candidates,
        accepted_matches, speaker_role_mapping_strict_matches, speaker_role_mapping,
        segment_level_proxy_textual,
    )
    display(textual_proxy_metrics_summary)

    result_frames = {
        "segment_level_proxy_groundtruth": segment_level_proxy_textual,
        "speaker_role_mapping_textual": speaker_role_mapping,
        "speaker_role_mapping_strict_matches": speaker_role_mapping_strict_matches,
        "speaker_role_evidence_aggregated": speaker_role_evidence_aggregated,
        "speaker_role_score_matrix_aggregated": speaker_role_score_matrix,
        "speaker_role_mapping_audit_aggregated": speaker_role_mapping_audit,
        "text_alignment_candidates": text_alignment_candidates,
        "text_alignment_matches": accepted_matches,
        "text_alignment_threshold_sensitivity": threshold_sensitivity,
        "textual_proxy_metrics_summary": textual_proxy_metrics_summary,
        "alignment_processing_summary": alignment_processing_summary,
        "holdout_role_mapping_metrics": holdout_metrics,
        "holdout_role_mapping_predictions": holdout_predictions,
        "metadata_join_audit_by_audio": seg_match_audit,
    }
    saved = mgp.save_and_upload_outputs(
        result_frames, PROXY_GROUNDTRUTH_DIR, gcs_client,
        GCS_PROXY_GROUNDTRUTH_PREFIX, upload_to_gcs=UPLOAD_TO_GCS,
    )
    print("\nArchivos guardados:", len(saved))
    print("Fase 06 completada.")
else:
    print("Fase 06 reutilizada desde outputs previos.")

# Incondicional: asegura que lo que hay en local también esté en GCS, tanto
# si se ejecutó la fase como si se saltó por tener outputs locales completos
# (caso típico: un compañero corrió esto en local y nunca lo subió).
mgp.ensure_phase_outputs_in_gcs(gcs_client, PROXY_GROUNDTRUTH_DIR, GCS_PROXY_GROUNDTRUTH_PREFIX)

Fase 06 reutilizada desde outputs previos.
Subiendo outputs 3/49: bq_metadata_notebook00_exact_with_transcripcion.csv
Subiendo outputs 4/49: checkpoints/text_alignment_candidates_checkpoint_0050.csv
Subiendo outputs 5/49: checkpoints/text_alignment_candidates_checkpoint_0100.csv
Subiendo outputs 6/49: checkpoints/text_alignment_candidates_checkpoint_0150.csv
Subiendo outputs 7/49: checkpoints/text_alignment_candidates_checkpoint_0200.csv
Subiendo outputs 8/49: checkpoints/text_alignment_candidates_checkpoint_0250.csv
Subiendo outputs 9/49: checkpoints/text_alignment_candidates_checkpoint_0300.csv
Subiendo outputs 10/49: checkpoints/text_alignment_candidates_checkpoint_0350.csv
Subiendo outputs 11/49: checkpoints/text_alignment_candidates_checkpoint_0400.csv
Subiendo outputs 12/49: checkpoints/text_alignment_candidates_checkpoint_0450.csv
Subiendo outputs 13/49: checkpoints/text_alignment_candidates_checkpoint_0500.csv
Subiendo outputs 14/49: checkpoints/text_alignment_candidates_checkp